In [0]:
import mlflow
import mlflow.xgboost
import xgboost as xgb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

warnings.filterwarnings("ignore")
plt.switch_backend("Agg")

In [0]:
# -------------------------------------------------------
# 1. MLflow Setup
# -------------------------------------------------------

mlflow.set_experiment("/Users/evansavo@gmail.com/air_quality_forecast_exp")
mlflow.set_registry_uri("databricks-uc")

UC_MODEL_NAME = "air_quality.03_gold.pm25_forecast_model"

In [0]:
# -------------------------------------------------------
# 2. Load Data
# -------------------------------------------------------

df = spark.read.table("air_quality.`03_gold`.ml_features_table").toPandas()

# Filter valid target rows
df = df[df["target_next_30d_pm25_median"].notnull()]

In [0]:
# -------------------------------------------------------
# 3. Feature Selection
# -------------------------------------------------------

FEATURES = [

    # temporal
    "year",
    "month",
    "day_of_week",
    "quarter",
    "is_weekend",

    # seasonal
    "is_winter",
    "is_spring",
    "is_summer",
    "is_fall",

    # infrastructure
    "population",
    "n_stations",
    "population_per_station",
    "monitoring_density_score",

    # geospatial
    "lat",
    "lon",

    # pollution trends
    "pm25_7d_avg",
    "pm25_30d_avg",
    "pm25_90d_avg",
    "pm10_30d_avg",
    "no2_30d_avg",

    # volatility
    "pm25_7d_volatility",

    # lag features
    "pm25_lag_1d",
    "pm25_lag_7d",
    "pm25_lag_14d",
    "pm25_lag_30d",

    # range metrics
    "pm25_30d_min",
    "pm25_30d_max",
    "pm25_30d_range",

    # trend indicators
    "pm25_trend",
    "pm25_recent_ratio",

    # historical exceedance
    "was_exceeding_who_7d",
    "was_exceeding_who_30d",
]

TARGET = "target_next_30d_pm25_median"

X = df[FEATURES].astype("float32")
y = df[TARGET].astype("float32")

In [0]:
# -------------------------------------------------------
# 4. Train Test Split
# -------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [0]:
# -------------------------------------------------------
# 5. Train Model with MLflow
# -------------------------------------------------------

with mlflow.start_run(run_name="pm25_forecast_xgboost") as run:

    params = {
        "max_depth": 6,
        "learning_rate": 0.05,
        "n_estimators": 1000,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "random_state": 42,
        "objective": "reg:squarederror",
        "tree_method": "hist"
    }

    mlflow.log_params(params)

    model = xgb.XGBRegressor(**params)

    model.fit(
        X_train,
        y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )


    # -------------------------------------------------------
    # 6. Predictions
    # -------------------------------------------------------

    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    metrics = {
        "mae": mae,
        "rmse": rmse,
        "r2": r2
    }

    mlflow.log_metrics(metrics)


    # -------------------------------------------------------
    # 7. Feature Importance Artifact
    # -------------------------------------------------------

    importance = model.feature_importances_

    fi = pd.DataFrame({
        "feature": FEATURES,
        "importance": importance
    }).sort_values("importance", ascending=False)

    fi.to_csv("feature_importance.csv", index=False)

    mlflow.log_artifact("feature_importance.csv")


    # -------------------------------------------------------
    # 8. Prediction Scatter Plot
    # -------------------------------------------------------

    fig = plt.figure()

    plt.scatter(y_test, y_pred)
    plt.xlabel("Actual PM2.5")
    plt.ylabel("Predicted PM2.5")
    plt.title("Predicted vs Actual")

    mlflow.log_figure(fig, "prediction_vs_actual.png")

    plt.close()


    # -------------------------------------------------------
    # 9. Residual Plot
    # -------------------------------------------------------

    residuals = y_test - y_pred

    fig = plt.figure()

    plt.hist(residuals, bins=40)
    plt.title("Residual Distribution")

    mlflow.log_figure(fig, "residuals.png")

    plt.close()


    # -------------------------------------------------------
    # 10. Feature Importance Plot
    # -------------------------------------------------------

    fig = plt.figure()

    sorted_idx = np.argsort(importance)

    plt.barh(
        range(len(sorted_idx)),
        importance[sorted_idx]
    )

    plt.yticks(
        range(len(sorted_idx)),
        [FEATURES[i] for i in sorted_idx]
    )

    plt.title("Feature Importance")

    mlflow.log_figure(fig, "feature_importance.png")

    plt.close()


    # -------------------------------------------------------
    # 11. 90-Day Forecast Artifact
    # -------------------------------------------------------

    forecast_df = df.copy()

    forecast_df["forecast_90d_pm25"] = (
        forecast_df["pm25_30d_avg"]
        + (forecast_df["pm25_trend"] * 3)
    )

    forecast_df[
        ["city", "country", "forecast_90d_pm25"]
    ].to_csv("city_forecast_90d.csv", index=False)

    mlflow.log_artifact("city_forecast_90d.csv")


    # -------------------------------------------------------
    # 12. Log Model
    # -------------------------------------------------------

    signature = mlflow.models.infer_signature(
        X_train.iloc[:5],
        model.predict(X_train.iloc[:5])
    )

    mlflow.xgboost.log_model(
        model,
        artifact_path="model",
        registered_model_name=UC_MODEL_NAME,
        signature=signature,
        input_example=X_train.iloc[:5]
    )

    print(f"Run ID: {run.info.run_id}")
    print(f"Model registered: {UC_MODEL_NAME}")


# -------------------------------------------------------
# 13. Model Registry Alias
# -------------------------------------------------------

client = mlflow.tracking.MlflowClient()

latest_version = client.search_model_versions(
    f"name='{UC_MODEL_NAME}'"
)[0].version

client.set_registered_model_alias(
    name=UC_MODEL_NAME,
    alias="champion",
    version=latest_version
)

print(f"Champion model set to version {latest_version}")

In [0]:
# -------------------------------------------------------
# 5. Hyperparameter Search (Databricks-safe)
# -------------------------------------------------------

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    "max_depth":        randint(3, 10),
    "learning_rate":    uniform(0.01, 0.2),
    "subsample":        uniform(0.6, 0.4),
    "colsample_bytree": uniform(0.5, 0.5),
    "min_child_weight": randint(1, 10),
    "gamma":            uniform(0, 0.5),
}

# n_estimators handled by early stopping — remove it from param_dist
base_model = xgb.XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    n_estimators=1000,        # high ceiling — early stopping will cut it short
    early_stopping_rounds=5, # stops if no improvement for 30 rounds
    eval_metric="mae",
    random_state=42
)

search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=5,
    scoring="neg_mean_absolute_error",
    cv=2,
    verbose=1,
    n_jobs=1,            # must be 1 on Databricks driver
    random_state=42
)

# Pass eval set so early stopping works inside CV
fit_params = {
    "eval_set": [(X_test, y_test)],
    "verbose": False
}

search.fit(X_train, y_train, **fit_params)

best_params = search.best_params_
model = search.best_estimator_

print(f"Best params: {best_params}")
print(f"Best CV MAE: {-search.best_score_:.4f}")

# -------------------------------------------------------
# 6. Train Model with MLflow
# -------------------------------------------------------

with mlflow.start_run(run_name="pm25_forecast_xgboost") as run:

    # Log best params + CV score from search
    mlflow.log_params(best_params)
    mlflow.log_metric("cv_mae", -search.best_score_)

    # model is already fitted by search — go straight to predictions
    y_pred = model.predict(X_test)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    mlflow.log_metrics({"mae": mae, "rmse": rmse, "r2": r2})


    # -------------------------------------------------------
    # 7. Feature Importance Artifact
    # -------------------------------------------------------

    importance = model.feature_importances_

    fi = pd.DataFrame({
        "feature": FEATURES,
        "importance": importance
    }).sort_values("importance", ascending=False)

    fi.to_csv("feature_importance.csv", index=False)
    mlflow.log_artifact("feature_importance.csv")


    # -------------------------------------------------------
    # 8. Prediction Scatter Plot
    # -------------------------------------------------------

    fig = plt.figure()
    plt.scatter(y_test, y_pred)
    plt.xlabel("Actual PM2.5")
    plt.ylabel("Predicted PM2.5")
    plt.title("Predicted vs Actual")
    mlflow.log_figure(fig, "prediction_vs_actual.png")
    plt.close()
    plt.show()


    # -------------------------------------------------------
    # 9. Residual Plot
    # -------------------------------------------------------

    residuals = y_test - y_pred

    fig = plt.figure()
    plt.hist(residuals, bins=40)
    plt.title("Residual Distribution")
    mlflow.log_figure(fig, "residuals.png")
    plt.close()
    plt.show()


    # -------------------------------------------------------
    # 10. Feature Importance Plot
    # -------------------------------------------------------

    sorted_idx = np.argsort(importance)

    fig = plt.figure()
    plt.barh(range(len(sorted_idx)), importance[sorted_idx])
    plt.yticks(range(len(sorted_idx)), [FEATURES[i] for i in sorted_idx])
    plt.title("Feature Importance")
    mlflow.log_figure(fig, "feature_importance.png")
    plt.close()
    plt.show()


    # -------------------------------------------------------
    # 11. 90-Day Forecast Artifact
    # -------------------------------------------------------

    forecast_df = df.copy()
    forecast_df["forecast_90d_pm25"] = (
        forecast_df["pm25_30d_avg"] + (forecast_df["pm25_trend"] * 3)
    )
    forecast_df[["city", "country", "forecast_90d_pm25"]].to_csv(
        "city_forecast_90d.csv", index=False
    )
    mlflow.log_artifact("city_forecast_90d.csv")


    # -------------------------------------------------------
    # 12. Log Model
    # -------------------------------------------------------

    signature = mlflow.models.infer_signature(
        X_train.iloc[:5],
        model.predict(X_train.iloc[:5])
    )

    mlflow.xgboost.log_model(
        model,
        artifact_path="model",
        registered_model_name=UC_MODEL_NAME,
        signature=signature,
        input_example=X_train.iloc[:5]
    )

    print(f"Run ID: {run.info.run_id}")
    print(f"Model registered: {UC_MODEL_NAME}")


# -------------------------------------------------------
# 13. Model Registry Alias
# -------------------------------------------------------

client = mlflow.tracking.MlflowClient()

latest_version = client.search_model_versions(
    f"name='{UC_MODEL_NAME}'"
)[0].version

client.set_registered_model_alias(
    name=UC_MODEL_NAME,
    alias="champion",
    version=latest_version
)

print(f"Champion model set to version {latest_version}")